In [ ]:
# lesson 1: text in & text out
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    temperature=temperature,
    n_ctx=n_ctx,
    verbose=False,
)
max_tokens = max_tokens


def generate(prompt: str) -> str:
    """
    Generate text from a prompt.
    
    Args:
        prompt: The input text prompt
        temperature: Optional temperature override
        stop: Optional list of stop sequences
        
    Returns:
        Generated text as a string
    """
    kwargs = {
        "prompt": prompt,
        "max_tokens": self.max_tokens
    }
    
    response = llm(**kwargs)
    return response["choices"][0]["text"].strip()

input = 
print("llm say:",generate(input))

In [6]:
# use zhipu glm model

from zhipuai import ZhipuAI

# 初始化客户端
client = ZhipuAI(api_key="6c33937d2e5d461297fa7d0736fff6ec.dZfTpqiXX7xDR65G")  # 替换为你的 API Key

def generate(prompt: str) -> str:
    """
    调用智谱 GLM 模型生成文本
    
    Args:
        prompt: 输入的提示词
        
    Returns:
        模型生成的文本
    """
    response = client.chat.completions.create(
        model="glm-4.7",  # 可选: glm-4, glm-4-plus, glm-4-flash, glm-4.7 等
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,      # 控制随机性，范围 0-1
        max_tokens=4096,      # 最大生成长度
        top_p=0.7,            # 核采样参数
    )
    
    return response.choices[0].message.content.strip()

# 使用示例
input_text = "你好，请介绍一下自己"
print("LLM say:", generate(input_text))

LLM say: 你好！我是由Z.ai开发的GLM大语言模型，我的设计目的是通过自然语言处理技术为用户提供信息和帮助。

我被训练用于理解问题、生成文本、回答知识和提供创意建议，不过我也有知识边界，无法进行实时互动或执行物理操作。我致力于以尊重、有益的方式与用户交流，同时保护用户隐私。

有什么我能帮助你解答的问题或提供协助的领域吗？


In [8]:
# lesson 2: systerm prompt

# system_prompt = (
#             "You are a calm, precise, and helpful AI assistant. "
#             "You explain concepts simply and avoid unnecessary jargon. "
#             "You are honest about what you know and don't know."
#         )

system_prompt = (
            "You are a professional physics expert. "
            "You explain concepts professionally and enough understandable. "
            "You are honest about what you know and don't know."
            "You can use physics formula to explain concept"
        )

def generate_with_role(user_input: str) -> str:
    """
    Generate with a system prompt to shape behavior.
    """
    # Use a format that doesn't confuse the model
    prompt = f"""{system_prompt}

User: {user_input}
Assistant:"""
    
    response = generate(prompt)
    # Clean up any potential tag artifacts
    response = response.replace('<SYSTEM>', '').replace('</SYSTEM>', '')
    response = response.replace('<USER>', '').replace('</USER>', '')
    return response.strip()

input_text = "please explain quantum field theory, and introduce basic principles"
print("LLM say:", generate_with_role(input_text))

LLM say: Hello. As a physics expert, I am happy to explain Quantum Field Theory (QFT). It is arguably the most successful and precise framework we have for describing the fundamental building blocks of nature.

At its core, **Quantum Field Theory is the combination of classical field theory, special relativity, and quantum mechanics.**

Before QFT, we had two separate rulebooks:
1.  **Quantum Mechanics:** Described small particles (like electrons) but struggled with things moving very fast or with particles being created or destroyed.
2.  **Special Relativity:** Described things moving fast but treated space and time as a smooth continuum, ignoring the "jumpy" nature of quantum particles.

QFT was born to solve this conflict. Here is the conceptual shift you need to make:

### The Fundamental Concept: Fields, Not Particles

In classical physics, we view particles as "balls" and fields as smooth backgrounds (like a magnetic field filling space).

**In QFT, we flip this upside down.**

W

In [ ]:
# lesson 3: answear with json format

import json
from zhipuai import ZhipuAI

# 初始化客户端
client = ZhipuAI(api_key="6c33937d2e5d461297fa7d0736fff6ec.dZfTpqiXX7xDR65G")  # 替换为你的 API Key

def generate(prompt: str) -> str:
    """
    调用智谱 GLM 模型生成文本
    
    Args:
        prompt: 输入的提示词
        
    Returns:
        模型生成的文本
    """
    response = client.chat.completions.create(
        model="glm-4.7",  # 可选: glm-4, glm-4-plus, glm-4-flash, glm-4.7 等
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,      # 控制随机性，范围 0-1
        max_tokens=4096,      # 最大生成长度
        top_p=0.7,            # 核采样参数
    )
    
    return response.choices[0].message.content.strip()

def safe_json_parse(text: str) -> dict | None:
    """
    Safely parse JSON text, returning None on failure.
    
    Args:
        text: String that might be valid JSON
        
    Returns:
        Parsed JSON as a dictionary, or None if parsing fails
    """
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

def extract_json_from_text(text: str) -> dict | None:
    """
    Try to extract JSON from text that might have extra content.
    
    This handles cases where the model adds explanations before/after JSON.
    
    Args:
        text: Text that might contain JSON
        
    Returns:
        Parsed JSON if found, None otherwise
    """
    if not text:
        return None
    
    # Clean up the text - remove common markdown code blocks
    text = text.strip()
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    text = text.strip()
    
    # Remove common prefixes that models sometimes add
    prefixes = ["JSON:", "Response:", "Answer:", "Here's the JSON:", "The JSON is:"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    
    # Try direct parsing first
    result = safe_json_parse(text)
    if result is not None:
        return result
    
    # Try to find JSON between curly braces (most common case)
    start = text.find('{')
    end = text.rfind('}')
    
    if start != -1 and end != -1 and end > start:
        json_text = text[start:end+1]
        result = safe_json_parse(json_text)
        if result is not None:
            return result
        
        # Try to fix common issues: unclosed strings, missing quotes
        # This is a simple heuristic - if we're close, try to fix it
        if json_text.count('"') % 2 != 0:
            # Odd number of quotes - try to close the last string
            last_quote = json_text.rfind('"')
            if last_quote > 0:
                # Check if it's an opening quote
                before = json_text[:last_quote]
                if before.count('"') % 2 == 0:
                    # This might be an unclosed string, try adding a closing quote
                    try_fix = json_text[:last_quote+1] + '"' + json_text[last_quote+1:] + '}'
                    result = safe_json_parse(try_fix)
                    if result is not None:
                        return result
    
    # Try to find JSON between square brackets (for arrays)
    start = text.find('[')
    end = text.rfind(']')
    
    if start != -1 and end != -1 and end > start:
        json_text = text[start:end+1]
        result = safe_json_parse(json_text)
        if result is not None:
            return result
    
    # Last resort: try to extract key-value pairs from text
    # This is very heuristic and may not work well
    if '{' in text or '[' in text:
        # Try to find any JSON-like structure
        lines = text.split('\n')
        for line in lines:
            line = line.strip()
            if line.startswith('{') or line.startswith('['):
                result = safe_json_parse(line)
                if result is not None:
                    return result
    
    return None

system_prompt = (
            "You are a calm, precise, and helpful AI assistant. "
            "You explain concepts simply and avoid unnecessary jargon. "
            "You are honest about what you know and don't know."
        )

def generate_structured(user_input: str, schema: str) -> dict | None:
        """
        Generate structured JSON output with validation and retries.
        
        Lesson 03 version.
        
        Args:
            user_input: The user's question or request
            schema: JSON schema description
            
        Returns:
            Parsed JSON dictionary or None if all retries failed
        """
        prompt = f"""{system_prompt}

CRITICAL INSTRUCTIONS:
1. Respond with ONLY valid JSON
2. No explanations, no markdown, no extra text before or after the JSON
3. Start your response with {{ and end with }}

Schema you must follow:
{schema}

User request: {user_input}

Response (JSON only):"""
        
        # Try up to 3 times
        for attempt in range(3):
            response = generate(prompt)
            parsed = extract_json_from_text(response)
            
            if parsed is not None:
                print("LLM say: ", response)
                return parsed
        
        return None
    
schema = '''
{
  "topic": string,
  "difficulty": "beginner" | "intermediate" | "advanced"
  "answer": content of answering
}
'''

result = generate_structured(
    "Explain quantum computing",
    schema
)

print(result)

LLM say:  {
  "topic": "Quantum Computing",
  "difficulty": "beginner",
  "answer": "Quantum computing is a new type of computation that harnesses the strange properties of quantum physics to solve problems that are too complex for classical computers. While traditional computers use 'bits' (which are like light switches that are either on or off, represented as 0 or 1), quantum computers use 'qubits'. A qubit can be 0, 1, or both at the same time thanks to a property called 'superposition'. This allows quantum computers to process a vast number of possibilities simultaneously. Another key property is 'entanglement', which links qubits together so that the state of one instantly influences the state of another, no matter the distance between them. Because of these abilities, quantum computers have the potential to revolutionize fields like medicine, materials science, and cryptography by performing calculations that would take today's supercomputers millions of years to finish."
}
{'to

In [ ]:
# lession 4: decide

system_prompt = (
            "You are a calm, precise, and helpful AI assistant. "
            "You explain concepts simply and avoid unnecessary jargon. "
            "You are honest about what you know and don't know."
        )

def decide(user_input: str, choices: list[str]) -> str | None:
    """
    Make the model choose from a finite set of options.
    
    Lesson 04 version.
    
    Args:
        user_input: The input to make a decision about
        choices: List of possible actions/decisions
        
    Returns:
        The chosen action or None if decision failed
    """
    options = "\n".join(f"- {choice}" for choice in choices)
    
    prompt = f"""{system_prompt}

You must choose ONE of the following options. Respond with ONLY valid JSON.

CRITICAL INSTRUCTIONS:
1. Respond with ONLY valid JSON
2. No explanations, no markdown, no other text
3. Start your response with {{ and end with }}

Available choices:
{options}

Required JSON format:
{{"decision": "one_of_the_choices_above"}}

User request: {user_input}

Response (JSON only):"""
        
    for attempt in range(3):
        response = generate(prompt)
        parsed = extract_json_from_text(response)
        
        if parsed and "decision" in parsed:
            decision = parsed["decision"]
            if decision in choices:
                return decision
    
    return None

decision = decide(
    "Can you play a music for me?",
    choices=["answer_question", "summarize_text", "translate","understand_content","write_text_file","read_text_file","none_of_the_above"]
)

print(decision)
# Output: "summarize_text"


none_of_the_above


In [29]:
# lesson 5: tools

from typing import Any, Dict, List, Optional 
from typing import Any

def request_tool(user_input: str) -> dict | None:
    """
    Have the model request a tool call.
    
    Lesson 05 version.
    
    Args:
        user_input: The user's request
        
    Returns:
        Tool call specification or None if request failed
    """
    prompt = f"""{system_prompt}

You are a tool-calling assistant. When asked a question, respond with ONLY valid JSON.

Available tools:
1. calculator - Perform basic arithmetic
   Parameters: a (number), b (number), operation ("add", "subtract", "multiply", "divide")

2. get_weather - Get weather information for a city
   Parameters: city (string), unit (string, optional: "celsius" or "fahrenheit", default "celsius")

CRITICAL INSTRUCTIONS:
1. Respond with ONLY valid JSON
2. No explanations, no markdown, no other text

Examples:
- {{"tool": "calculator", "arguments": {{"a": 42, "b": 7, "operation": "multiply"}}}}
- {{"tool": "get_weather", "arguments": {{"city": "Beijing", "unit": "celsius"}}}}

User request: {user_input}

Response (JSON only):"""
    
    for attempt in range(3):
        response = generate(prompt)
        parsed = extract_json_from_text(response)
        
        if parsed and "tool" in parsed and "arguments" in parsed:
            return parsed
    
    return None

def execute_tool_call(tool_call: dict) -> Any:
    """
    Execute a tool call requested by the model.
    
    Args:
        tool_call: Dictionary with "tool" and "arguments"
        
    Returns:
        Result of the tool execution
    """
    return execute_tool(tool_call["tool"], tool_call["arguments"])

# lesson 5: tools - 完整版本

from typing import Any, Dict, List, Optional 
import json

# ============ 工具定义 ============

def calculator(a: float, b: float, operation: str) -> Any:
    """
    计算器工具
    
    Args:
        a: 第一个数字
        b: 第二个数字
        operation: 运算类型 ("add", "subtract", "multiply", "divide")
    
    Returns:
        计算结果或错误信息
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            return "Error: Division by zero"
        return a / b
    else:
        return f"Error: Unknown operation '{operation}'"

def get_weather(city: str, unit: str = "celsius") -> str:
    """
    获取天气信息（示例）
    
    Args:
        city: 城市名称
        unit: 温度单位 ("celsius" 或 "fahrenheit")
    
    Returns:
        天气信息字符串
    """
    # 实际使用时替换为真实 API 调用
    weather_data = {
        "Beijing": {"temp": 25, "condition": "Sunny"},
        "Shanghai": {"temp": 22, "condition": "Cloudy"},
        "New York": {"temp": 18, "condition": "Rainy"}
    }
    
    if city not in weather_data:
        return f"Sorry, no weather data for {city}"
    
    weather = weather_data[city]
    temp = weather["temp"]
    
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    
    return f"{city}: {temp}°{'F' if unit == 'fahrenheit' else 'C'}, {weather['condition']}"

# 工具注册表
TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather,
}

# ============ 工具执行函数 ============

def execute_tool(tool_name: str, arguments: dict) -> Any:
    """
    Execute the requested tool with given arguments.
    
    Args:
        tool_name: Name of the tool to execute
        arguments: Dictionary of arguments for the tool
        
    Returns:
        Result of the tool execution
    """
    if tool_name not in TOOLS:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"
    
    tool_func = TOOLS[tool_name]
    
    try:
        # 使用 **arguments 解包参数
        result = tool_func(**arguments)
        return result
    except TypeError as e:
        return f"Error: Invalid arguments for {tool_name}. Expected: {tool_func.__doc__}"
    except Exception as e:
        return f"Error executing {tool_name}: {str(e)}"

def execute_tool_call(tool_call: dict) -> Any:
    """
    Execute a tool call requested by the model.
    
    Args:
        tool_call: Dictionary with "tool" and "arguments"
        
    Returns:
        Result of the tool execution
    """
    return execute_tool(tool_call["tool"], tool_call["arguments"])


# tool_call = request_tool("What is 42 - 7?")
tool_call = request_tool("What the weather in the Beijing")
print(f"Tool request: {tool_call}")

if tool_call:
    result = execute_tool_call(tool_call)
    print(f"Tool result: {result}")

Tool request: {'tool': 'get_weather', 'arguments': {'city': 'Beijing'}}
Tool result: Beijing: 25°C, Sunny


In [ ]:
# lesson 6: agent loop (enhanced with tools)

system_prompt = (
    "You are a calm, precise, and helpful AI assistant. "
    "You explain concepts simply and avoid unnecessary jargon. "
    "You are honest about what you know and don't know."
)

# ============ New tools ============

def read_file(path: str) -> str:
    """Read a local text file and return its content."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File '{path}' not found."
    except Exception as e:
        return f"Error reading '{path}': {e}"

def write_file(path: str, content: str) -> str:
    """Write content to a local text file."""
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Successfully wrote to '{path}'."
    except Exception as e:
        return f"Error writing to '{path}': {e}"

def web_search(query: str) -> str:
    """Search the web and return a summary of results."""
    try:
        import subprocess
        # result = subprocess.run(
        #     ["curl", "-s", f"https://api.duckduckgo.com/?q={query}&format=json&no_html=1"],
        #     capture_output=True, text=True, timeout=10
        # )
        result = subprocess.run(
                ["curl", "-s", f"{query}"],
                capture_output=True, text=True, timeout=10
        )
        import json
        data = json.loads(result.stdout)
        abstract = data.get("Abstract", "")
        if abstract:
            return abstract
        related = data.get("RelatedTopics", [])
        summaries = []
        for item in related[:3]:
            if isinstance(item, dict) and "Text" in item:
                summaries.append(item["Text"])
        return "\n".join(summaries) if summaries else f"No results found for '{query}'."
    except Exception as e:
        return f"Error searching '{query}': {e}"

# Reuse tools from lesson 5
TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather,
    "read_file": read_file,
    "write_file": write_file,
    "web_search": web_search,
}

# ============ tool excution ============

def execute_tool(tool_name: str, arguments: dict) -> Any:
    """
    Execute the requested tool with given arguments.
    
    Args:
        tool_name: Name of the tool to execute
        arguments: Dictionary of arguments for the tool
        
    Returns:
        Result of the tool execution
    """
    if tool_name not in TOOLS:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"
    
    tool_func = TOOLS[tool_name]
    
    try:
        # 使用 **arguments 解包参数
        result = tool_func(**arguments)
        return result
    except TypeError as e:
        return f"Error: Invalid arguments for {tool_name}. Expected: {tool_func.__doc__}"
    except Exception as e:
        return f"Error executing {tool_name}: {str(e)}"


# ============ Tool description for prompt ============

TOOL_DESCRIPTION = """
1. calculator - Perform basic arithmetic
   Parameters: a (number), b (number), operation ("add", "subtract", "multiply", "divide")
2. get_weather - Get weather for a city
   Parameters: city (string), unit (string, optional: "celsius" or "fahrenheit")
3. read_file - Read a local text file
   Parameters: path (string)
4. write_file - Write content to a local file
   Parameters: path (string), content (string)
5. web_search - Search the web for information
   Parameters: query (string)
"""

# ============ Agent step ============

def agent_step(user_input: str, state: dict) -> dict | None:
    """
    One step of the agent loop: think, use a tool, answer, or signal done.

    Args:
        user_input: Current context (original question + prior tool results)
        state: {"steps": int, "done": bool, "history": list}

    Returns:
        Action dict or None if all retries failed
    """
    history_str = ""
    for h in state.get("history", []):
        history_str += f"- {h}\n"

    prompt = f"""{system_prompt}

You are an agent with access to tools. Decide the next action and respond with ONLY valid JSON.

Current step: {state.get('steps', 0)}
Conversation history:
{history_str if history_str.strip() else "(none)"}

Available tools:
{TOOL_DESCRIPTION}

Available actions:
- think: Reason about the problem without using tools. JSON: {{"action": "think", "thought": "your reasoning"}}
- tool_use: Call a tool. JSON: {{"action": "tool_use", "tool": "tool_name", "arguments": {{...}}}}
- answer: Give the final answer. JSON: {{"action": "answer", "text": "your answer"}}
- done: End the loop. JSON: {{"action": "done"}}

CRITICAL INSTRUCTIONS:
1. Respond with ONLY valid JSON, no extra text
2. If you need information or computation, use tool_use
3. When you have enough info, use answer
4. Start with {{ and end with }}

User request: {user_input}

Response (JSON only):"""

    for attempt in range(3):
        response = generate(prompt)
        parsed = extract_json_from_text(response)

        if parsed and "action" in parsed:
            state['steps'] = state.get('steps', 0) + 1
            return parsed

    return None

# ============ Run loop ============

def run_loop(user_input: str, max_steps: int = 8):
    """
    Run the agent loop with tool execution.

    Args:
        user_input: Initial user input
        max_steps: Maximum number of steps

    Returns:
        List of step results (each includes action, result, etc.)
    """
    state = {"steps": 0, "done": False, "history": []}
    results = []
    context = user_input

    while not state.get('done', False) and state.get('steps', 0) < max_steps:
        action = agent_step(context, state)

        if not action:
            results.append({"action": "error", "detail": "Failed to get a valid action"})
            break

        step_result = {"step": state['steps'], "action": action.get("action")}

        act = action.get("action")

        if act == "done":
            state['done'] = True
            step_result["detail"] = "Agent finished."

        elif act == "think":
            thought = action.get("thought", "")
            step_result["thought"] = thought
            state["history"].append(f"Thought: {thought}")

        elif act == "tool_use":
            tool_name = action.get("tool", "")
            arguments = action.get("arguments", {})
            step_result["tool"] = tool_name
            step_result["arguments"] = arguments

            if tool_name in TOOLS:
                tool_result = execute_tool(tool_name, arguments)
            else:
                tool_result = f"Error: Unknown tool '{tool_name}'"

            step_result["tool_result"] = tool_result
            state["history"].append(f"Used tool {tool_name}({arguments}) → {tool_result}")
            context = f"{user_input}\n\n[Tool result from {tool_name}]: {tool_result}"

        elif act == "answer":
            text = action.get("text", "")
            step_result["answer"] = text
            state["history"].append(f"Answer: {text}")
            state['done'] = True

        else:
            step_result["detail"] = f"Unknown action: {act}"

        results.append(step_result)

    return results

# ============ Demo ============

print("=" * 60)
print("Demo 1: Calculator question")
print("=" * 60)
results = run_loop("What is 1234 multiplied by 456 ?", max_steps=5)
for r in results:
    print(f"  Step {r.get('step', '?')}: {r.get('action', '?')}", end="")
    if "thought" in r:
        print(f" → {r['thought']}")
    elif "tool_result" in r:
        print(f" → tool={r['tool']} → {r['tool_result']}")
    elif "answer" in r:
        print(f" → {r['answer']}")
    else:
        print()

print()
print("=" * 60)
print("Demo 2: File read question")
print("=" * 60)
results = run_loop("Please read file agent/agent.py, tell me how many class in python file ", max_steps=5)
for r in results:
    print(f"  Step {r.get('step', '?')}: {r.get('action', '?')}", end="")
    if "thought" in r:
        print(f" → {r['thought']}")
    elif "tool_result" in r:
        print(f" → tool={r['tool']} → {str(r['tool_result'])[:100]}...")
    elif "answer" in r:
        print(f" → {r['answer'][:200]}...")
    else:
        print()


print()
print("=" * 60)
print("Demo 3: File write question")
print("=" * 60)
results = run_loop("Please create a txt file, input hello world", max_steps=5)
for r in results:
    print(f"  Step {r.get('step', '?')}: {r.get('action', '?')}", end="")
    if "thought" in r:
        print(f" → {r['thought']}")
    elif "tool_result" in r:
        print(f" → tool={r['tool']} → {str(r['tool_result'])[:100]}...")
    elif "answer" in r:
        print(f" → {r['answer'][:200]}...")
    else:
        print()
        
# print()
# print("=" * 60)
# print("Demo 4: Web search")
# print("=" * 60)
# results = run_loop("Please search https://time.is/zh/China 查看当前北京时间", max_steps=5)
# for r in results:
#     print(f"  Step {r.get('step', '?')}: {r.get('action', '?')}", end="")
#     if "thought" in r:
#         print(f" → {r['thought']}")
#     elif "tool_result" in r:
#         print(f" → tool={r['tool']} → {str(r['tool_result'])[:100]}...")
#     elif "answer" in r:
#         print(f" → {r['answer'][:200]}...")
#     else:
#         print()

Demo 1: Calculator question
  Step 1: tool_use → tool=calculator → 562704
  Step 2: answer → 1234 multiplied by 456 is 562,704.

Demo 2: File read question
  Step 1: tool_use → tool=read_file → """
The Agent - This file grows across all 10 lessons.

This is the heart of the repository. Each le...
  Step 2: answer → There is 1 class defined in agent/agent.py: the Agent class....

Demo 3: File write question
  Step 1: tool_use → tool=write_file → Successfully wrote to 'hello.txt'....
  Step 2: answer → I've already created the file 'hello.txt' with the content 'hello world'. The task is complete....


In [43]:
# lesson 7: memory control class

class Memory:
    """
    Simple memory storage for the agent.
    
    This is intentionally basic and grows in lessons:
    - Lesson 07: Basic list-based memory
    - Future: Add semantic search, persistence, etc.
    """
    
    def __init__(self):
        """Initialize empty memory."""
        self.items = []
    
    def add(self, item: str):
        """
        Add an item to memory.
        
        Args:
            item: String to remember
        """
        if item and item not in self.items:
            self.items.append(item)
    
    def get_all(self) -> list[str]:
        """
        Retrieve all memory items.
        
        Returns:
            List of all stored items
        """
        return self.items.copy()
    
    def get_recent(self, n: int = 5) -> list[str]:
        """
        Get the n most recent memory items.
        
        Args:
            n: Number of recent items to retrieve
            
        Returns:
            List of recent items
        """
        return self.items[-n:] if self.items else []
    
    def search(self, query: str) -> list[str]:
        """
        Simple search through memory items.
        
        Args:
            query: String to search for
            
        Returns:
            List of items containing the query
        """
        query_lower = query.lower()
        return [item for item in self.items if query_lower in item.lower()]
    
    def clear(self):
        """Clear all memory."""
        self.items = []
    
    def __len__(self) -> int:
        """Return the number of items in memory."""
        return len(self.items)
    
    def __repr__(self) -> str:
        """String representation of memory."""
        return f"Memory({len(self.items)} items)"

In [46]:
# lesson 7: memory

memory = Memory()

def memory_step(user_input: str, state: dict) -> dict | None:
    """
    One step of the memory-enabled agent loop.

    Args:
        user_input: Current context
        state: {"steps": int, "done": bool, "history": list}

    Returns:
        Action dict or None
    """
    memory_context = memory.get_all()

    if memory_context:
        memory_str = "You remember the following:\n" + "\n".join(
            f"- {item}" for item in memory_context
        )
    else:
        memory_str = "You have no memories yet."

    history_str = ""
    for h in state.get("history", []):
        history_str += f"- {h}\n"

    prompt = f"""{system_prompt}

You are an agent with memory. You must respond with ONLY valid JSON.

{memory_str}

CRITICAL INSTRUCTIONS:
1. Respond with ONLY valid JSON, no extra text
2. If the user tells you personal info (name, preferences, etc.), save it via save_to_memory
3. If the user asks about something you remember, USE YOUR MEMORY to answer
4. Start with {{ and end with }}
5. Save anything that you think it's important for tasks

Available actions:
- think: Reason before responding. JSON: {{"action": "think", "thought": "your reasoning"}}
- answer: Reply to user. JSON: {{"action": "answer", "reply": "your response", "save_to_memory": "fact to remember" or null}}
- done: End. JSON: {{"action": "done"}}

Examples:
- User: "My name is Alice" -> {{"action": "answer", "reply": "Nice to meet you, Alice!", "save_to_memory": "User's name is Alice"}}
- User: "What's my name?" (memory: "User's name is Alice") -> {{"action": "answer", "reply": "Your name is Alice!", "save_to_memory": null}}
- User: "I like pizza and sushi" -> {{"action": "answer", "reply": "Noted! Pizza and sushi, great taste!", "save_to_memory": "User likes pizza and sushi"}}

User input: {user_input}

Response (JSON only):"""

    for attempt in range(3):
        response = generate(prompt)
        parsed = extract_json_from_text(response)

        if parsed and "action" in parsed:
            state["steps"] = state.get("steps", 0) + 1
            return parsed

    return None

def run_memory_loop(user_input: str, max_steps: int = 5) -> list:
    """
    Run the memory-only agent loop.

    Args:
        user_input: User's message
        max_steps: Max steps before forced stop

    Returns:
        List of step results
    """
    state = {"steps": 0, "done": False, "history": []}
    results = []
    context = user_input

    while not state.get("done", False) and state.get("steps", 0) < max_steps:
        action = memory_step(context, state)

        if not action:
            results.append({"step": state["steps"], "action": "error", "detail": "Failed to parse action"})
            break

        act = action.get("action")
        step_result = {"step": state["steps"], "action": act}

        if act == "done":
            state["done"] = True
            step_result["detail"] = "Agent finished."

        elif act == "think":
            thought = action.get("thought", "")
            step_result["thought"] = thought
            state["history"].append(f"Thought: {thought}")

        elif act == "answer":
            reply = action.get("reply", "")
            save = action.get("save_to_memory")

            if save:
                memory.add(save)
                state["history"].append(f"Saved to memory: {save}")

            state["history"].append(f"Reply: {reply}")
            step_result["reply"] = reply
            step_result["saved"] = save
            state["done"] = True

        results.append(step_result)

    return results

# ============ Demo ============

conversations = [
    "Hi! My name is Yoru and I'm a software engineer.",
    "I really like playing rhythm games and reading sci-fi novels.",
    "Do you remember my name and what I do?",
    "What are my hobbies?",
    "Do you know what i'm doing now?",
    "Please tell me whats my name and give me a suggestion about my career plan"
]

for i, msg in enumerate(conversations, 1):
    print(f"{'='*60}")
    print(f"Turn {i}: {msg}")
    print(f"{'='*60}")
    results = run_memory_loop(msg, max_steps=3)
    for r in results:
        act = r.get("action", "?")
        print(f"  Step {r.get('step', '?')} [{act}]", end="")
        if "thought" in r:
            print(f" → {r['thought']}")
        elif "reply" in r:
            print(f" → {r['reply']}")
            if r.get("saved"):
                print(f"  📝 Saved: {r['saved']}")
        else:
            print()
    print(f"  Memory: {memory.get_all()}")
    print()

Turn 1: Hi! My name is Yoru and I'm a software engineer.
  Step 1 [answer] → Hello Yoru! Nice to meet you. It's great to connect with a software engineer. How can I help you today?
  📝 Saved: User's name is Yoru and they are a software engineer
  Memory: ["User's name is Yoru and they are a software engineer"]

Turn 2: I really like playing rhythm games and reading sci-fi novels.
  Step 1 [answer] → That sounds like a fun combination, Yoru! Rhythm games require great reflexes, and sci-fi novels offer amazing worlds to explore.
  📝 Saved: User likes playing rhythm games and reading sci-fi novels
  Memory: ["User's name is Yoru and they are a software engineer", 'User likes playing rhythm games and reading sci-fi novels']

Turn 3: Do you remember my name and what I do?
  Step 1 [answer] → Yes, I remember! Your name is Yoru and you're a software engineer.
  Memory: ["User's name is Yoru and they are a software engineer", 'User likes playing rhythm games and reading sci-fi novels']

Turn 4

In [ ]:
# lesson 8: plan creation 
